In [ ]:
import os
import torch
import torch.nn.functional as F
import json
import random

from jinyu_utils import jinyu_dataset
from jinyu_utils.jinyu_preprocess_wiki import parse_lines_with_index, merge_subdocs, PATTEN_REG_WIKI

from transformers import AutoTokenizer
from datasets import load_dataset, Dataset
from abc import ABC, abstractmethod

from torch.utils.data import DataLoader

from fastdllm_generate import add_gumbel_noise, get_num_transfer_tokens

from tqdm import tqdm
from modeling_llada_with_budget_and_cachekv import LLaDAModelLM

from datetime import datetime, timezone
from collections import defaultdict

def get_current_time_str():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")
# end


/home/exx/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:


'''initialize global constants'''

ID_TOKEN_MASK = 126336 # '|mdm_mask|'
ID_TOKEN_PADDING = 126081 # '|endoftext|'
ID_TOKEN_EOT = 126348 # '|eot_id|'

TYPES_REMASKING = {'truth_top_k', 'random_top_k'}



'''define token encoder function'''

class Tokenizer_(ABC):
    def __init__(self, tokenizer, len_max):
        self.tokenizer = tokenizer
        self.len_max = len_max
    # end

    @abstractmethod
    def _tokenize(self, ds_each):
        pass
    # end

    def __call__(self, ds_each):
        return self._tokenize(ds_each)
    # end
# end

class Tokenizer_wiki_simple(Tokenizer_):

    def _tokenize(self, ds_each):
        ids = self.tokenizer(
            ds_each['text'],
            add_special_tokens=False,               # avoids BOS/EOS being injected by tokenizer
            truncation=(self.len_max is not None),  # truncation and max_length is a pair
            max_length=self.len_max,
            # return_tensors='pt'
        )["input_ids"]


        return {
            'ids_input': ids,
            'length': len(ids)
        }
    # end tokenize
# end


class Collater_(ABC):
    def __init__(self, len_max, len_prompt, len_target, id_mask):
        self.len_max = len_max
        self.len_prompt = len_prompt
        self.len_target = len_target
        self.id_mask = id_mask
    # end

    @abstractmethod
    def _collate(self, ds_batch):
        pass
    # end

    def __call__(self, ds_batch):
        return self._collate(ds_batch)
    # end
# end


class Collater_wiki_simple(Collater_):

    def _collate(self, ds_batch):
        # batch: list of dicts with "input_ids" as python lists
        len_min = min(len(ds_each["ids_input"]) for ds_each in ds_batch)

        ids_input = torch.stack([torch.tensor(ds_each["ids_input"][:len_min], dtype=torch.long) for ds_each in ds_batch], dim=0) # [B, min_len]
        masks_input = torch.zeros_like(ids_input, dtype=bool)
        masks_input[:, self.len_prompt:] = True
        ids_target = torch.where(masks_input, ids_input, self.id_mask)
        ids_input[masks_input] = self.id_mask

        return {
            'ids_prompt_masked_full': ids_input,
            'ids_target_masked_full': ids_target
        }
    # end _collate
# end


class RefreshIdxHelper:
    TYPE_HIDDEN = {
        'k':'_k_previous',
        'v':'_v_previous'
    }

    def __init__(self, dict_filename_to_list_idx_sorted, type_hidden_str):
        self.dict_filename_to_list_idx_sorted = dict_filename_to_list_idx_sorted
        self.type_hidden=RefreshIdxHelper.TYPE_HIDDEN[type_hidden_str]
    # end

    def get_refresh_idx(self, x, id_sample, id_block, budget, return_sorted=True):
        filename = f'batch_{id_sample}{self.type_hidden}.pt'
        list_idx_sorted = self.dict_filename_to_list_idx_sorted[filename]

        if budget < 1.0:
            budget = int(len(list_idx_sorted[id_block]['idx']) * budget) or 1
        # end



        list_idx = torch.tensor(list_idx_sorted[id_block]['idx'], dtype=torch.long, device=x.device)
        list_value = torch.tensor(list_idx_sorted[id_block]['value_raw'], dtype=torch.float, device=x.device)

        idx_list_idx_rand = torch.randperm(list_idx.shape[0])
        idx_list_idx_rand_budget = idx_list_idx_rand[:budget]
        result = list_idx[idx_list_idx_rand_budget]

        print(f'[{id_sample}], [{id_block}]: {idx_list_idx_rand_budget}')

        return torch.sort(result)[0] if return_sorted else result
    # end
# end

class RefreshIdxHelper_Mocked(RefreshIdxHelper):

    POLICY_REFRESH_PREVIOUS_ALL = 'refresh_previous_all'
    POLICY_REFRESH_NONE = 'refresh_none'

    def __init__(self, type_refresh=None):
        self.type_refresh = type_refresh
    # end

    def get_refresh_idx(self, x, len_prompt=0):
        if self.type_refresh == RefreshIdxHelper_Mocked.POLICY_REFRESH_PREVIOUS_ALL:
            return torch.arange(len_prompt, dtype=torch.long, device=x.device)
        else:
            return torch.tensor([],dtype=torch.long, device=x.device)
        # end
    # end
# end

''' ppl calculation function'''
def calculate_ppl_and_conf(probs_all, mask_target, eps=1e-12):
    # probs_collected = probs_all[mask_target].reshape(mask_target.shape[0], -1)  # [B, K]
    probs_collected = probs_all[mask_target].reshape(-1)  # [B * K]

    # Arithmetic mean confidence (what you currently call mean_prob)
    mean_prob = probs_collected.mean(dim=-1)  # [B]

    # Per-token NLL and per-row PPL (geometric-mean based)
    nll_collected = -torch.log(probs_collected + eps)   # [B, K]
    nll_per = nll_collected.mean(dim=-1)                 # [B]
    ppl_per = torch.exp(nll_per)                        # [B]

    # Geometric mean confidence (this one is directly tied to PPL)
    # geo_prob = torch.exp(torch.log(probs_collected + eps).mean(dim=1))  # [B]
    # And ppl_per == 1 / geo_prob (up to eps effects)

    return ppl_per.item(), mean_prob.item()
# end


@ torch.no_grad()
def get_transfer_index(
    logits: torch.Tensor,
    temperature: float,
    remasking: str,
    mask_index: torch.Tensor,   # (B, L) bool
    x: torch.Tensor,            # (B, L) long
    y: torch.Tensor,            # (B, L) long
    num_transfer_tokens,        # (B,) or (B,1) long tensor, or None when threshold is used
    threshold: float = None,
):
    """
    Returns:
        x0: (B, L) long — proposed tokens
        transfer_index: (B, L) bool — which positions to update this step
    """
    # 1) Sample proposal x0
    # Gumbel-noise for exploration; if temperature==0, add_gumbel_noise should no-op
    logits_with_noise = add_gumbel_noise(logits, temperature=temperature)
    x0 = torch.argmax(logits_with_noise, dim=-1)  # (B, L), long

    # 2) Confidence for chosen tokens (or random)
    p = F.softmax(logits.to(torch.float64), dim=-1)
    x0_p = torch.gather(p, dim=-1, index=y.unsqueeze(-1)).squeeze(-1)  # (B, L), float64
    # x0_p = torch.rand(x0.shape, device=x0.device, dtype=torch.float64)  # (B, L)  # removed by jinyu

    # Only modify masked spots; keep others as original x and set their confidence to -inf
    # TODO: we have error here
    x0 = torch.where(mask_index, x0, x) # mask_index is only this block

    neg_inf = torch.tensor(torch.finfo(x0_p.dtype).min, device=x0.device, dtype=x0_p.dtype)
    confidence = torch.where(mask_index, x0_p, neg_inf)  # (B, L)   # so only the masked part has confidence

    # Ensure shape (B,) long    jinyu: re-calculate num_transfer_token every time(I think)
    if num_transfer_tokens.dim() == 2 and num_transfer_tokens.size(1) == 1:
        num_transfer_tokens = num_transfer_tokens.squeeze(1)
    # end

    num_transfer_tokens = num_transfer_tokens.to(dtype=torch.long, device=confidence.device)
    num_transfer_tokens = torch.clamp(num_transfer_tokens, min=0)   # jinyu: can it be negative???


    # Sort confidences descending (masked positions are valid; others are -inf)
    # idx: (B, L) gives positions in original sequence sorted by confidence
    if remasking == 'random_top_k':
        idx_sorted_random = torch.argsort(
            torch.where(
                mask_index,
                torch.rand(confidence.shape[0], confidence.shape[1], device=confidence.device),
                confidence
            ),
            dim=1,
            descending=True
        )
        idx_sorted = idx_sorted_random  # for your read
    elif remasking == 'truth_top_k':
        idx_sorted = torch.argsort(confidence, dim=1, descending=True)
    else:
        raise NotImplementedError()
    # end

    B, L = confidence.shape
    # Build a mask that is True for the first k[b] columns in each row (sorted order)
    cols = torch.arange(L, device=confidence.device).unsqueeze(0).expand(B, L)   # (B, L)
    k_expanded = num_transfer_tokens.unsqueeze(1).expand(B, L)                   # (B, L)
    select_sorted = cols < k_expanded                                            # (B, L) bool for top k

    # Scatter the sorted True/False back to original column order
    # Use integer scatter then cast to bool (scatter_ on bool can be finicky across versions)
    transfer_int = torch.zeros(B, L, device=confidence.device, dtype=torch.int8) # (B, L)
    transfer_int = transfer_int.scatter(1, idx_sorted, select_sorted.to(torch.int8))
    transfer_index = transfer_int.bool() & mask_index  # ensure we never select unmasked

    return x0, x0_p, transfer_index
# end

In [ ]:
@torch.no_grad()
def run_model_with_budget(
    model,
    ids_input_masked_full,
    ids_target_masked_full,
    len_prompt,
    remasking='truth_top_k',
    steps=128,
    gen_length=128,
    block_length=128,
    temperature=0.,
    mask_id=126336,
    is_eval=True,
    use_cache=False,
    id_batch=0,
    helper_refresh: RefreshIdxHelper = None,
    budget_refresh=8
):
    
    x, y = ids_input_masked_full, ids_target_masked_full
    B = x.shape[0]

    probs_all = torch.zeros(x.shape, dtype=torch.float64).to(model.device)

    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length

    assert steps % num_blocks == 0
    steps_per_block = steps // num_blocks

    past_key_values = None
    if use_cache:
        x_prompt = x[:, :len_prompt]
        idx_current = torch.arange(x_prompt.shape[1], device=x.device)
        out_current = model(x_prompt, use_cache=True, idx_current=idx_current)
        past_key_values = out_current.past_key_values
    # end


    for id_block in range(num_blocks):

        position_start = len_prompt + id_block * block_length
        position_end = position_start + block_length
        block_mask_index = (x[:, position_start:position_end] == mask_id)  # (B, block_length)
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps_per_block)  # (B, steps_per_block)

        for step_in_block in range(steps_per_block):
            # Evaluate logits only for current block with cache
            if (x[:, position_start:position_end] == mask_id).sum() == 0:
                break
            # end

            if  isinstance(helper_refresh, RefreshIdxHelper_Mocked):
                idx_refresh = helper_refresh.get_refresh_idx(x, position_start) # full prompt is to refresh
            else:
                # id_sample, id_block, budget, return_sorted=True
                idx_refresh = helper_refresh.get_refresh_idx(x, id_batch, id_block, budget_refresh, return_sorted=True)
            # end

            idx_denoising = torch.arange(position_start, position_end,dtype=torch.long, device=x.device)
            idx_current = torch.cat([idx_refresh, idx_denoising])

            x_current = x[:, idx_current]

            output_current = model(
                x_current,
                past_key_values=past_key_values,
                use_cache=use_cache,
                idx_current=idx_current,
                shape_target=(x.shape[0], position_end, -1)
            )

            # (B, [refresh|blk])
            logits_current = output_current.logits
            logits_blk = logits_current[:, -idx_denoising.shape[-1]:] # (B, [blk])
            past_key_values=output_current.past_key_values if use_cache else None # update past_key_values here

            # Mask and quota for this step (all tensor ops)
            mask_blk = (x[:, position_start:position_end] == mask_id)  # (B, block_length)
            blk_x = x[:, position_start:position_end]
            blk_y = y[:, position_start:position_end]
            blk_prob = probs_all[:, position_start:position_end]

            quota_i = num_transfer_tokens[:, step_in_block]  # (B,)
            blk_x0, blk_x0_p, transfer_idx_blk = get_transfer_index(
                logits_blk,
                temperature,
                remasking,
                mask_blk,
                blk_x,
                blk_y,
                quota_i
            )

            if is_eval:
                blk_x[transfer_idx_blk] = blk_y[transfer_idx_blk]
                blk_prob[transfer_idx_blk] = blk_x0_p[transfer_idx_blk]
            else:
                blk_x[transfer_idx_blk] = blk_x0[transfer_idx_blk]
            # end

    return probs_all, y != mask_id
# end

In [ ]:
'''load dataset first'''
name_dataset = jinyu_dataset.LIST_DATASET[1]
ds = load_dataset(*name_dataset, split='all')
docs, _ = parse_lines_with_index(PATTEN_REG_WIKI, ds['text'])
docs = docs['subdocs']

samples = []
for doc in docs:
    lines_1 = doc['texts']
    paragraph_1 = ' '.join(lines_1)
    lines_remain, titles = merge_subdocs(doc['subdocs'])
    paragraph_remain = ' '.join(lines_remain)
    prefix = paragraph_1
    target = paragraph_remain
    samples.append({'text': paragraph_1 + ' ' + paragraph_remain})
# end

ds_origin = Dataset.from_list(samples[:100])

'''initialize constant hyper-parameters'''
id_model_g = 'GSAI-ML/LLaDA-8B-Base'
id_mask_g = 126336
device_g = 'cuda:0'
size_batch_g = 1

'''load tokenizer'''
tokenizer = AutoTokenizer.from_pretrained(
    id_model_g,
    trust_remote_code=True
)

if tokenizer.padding_side != 'left':
    tokenizer.padding_side = 'left'
# end
assert tokenizer.pad_token_id != 126336


'''load model'''
model_kwargs = {}
model = LLaDAModelLM.from_pretrained(
    id_model_g,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    **model_kwargs
)

model = model.eval().to(device_g)


'''hyper parameter to be set'''
len_prompt_g = 128
len_target_g = 256
num_blocks_g = 4
num_unmask_per_iter_g = 1

'''hyper parameter can be calculated'''
len_max_g = len_prompt_g + len_target_g
size_block_g = int(len_target_g / num_blocks_g)
assert num_unmask_per_iter_g <= size_block_g
steps_g = int(len_target_g / num_unmask_per_iter_g)

'''use cache and refresh'''
use_cache_g = True
use_refresh_g = True
type_hidden_refresh_str_g = 'v'
path_file_data_refresh = 'all_in_one_sim_report_raw_value_abs_p.json'


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Loading checkpoint shards: 100%|██████████| 6/6 [00:01<00:00,  5.23it/s]


In [ ]:
if not use_cache_g: # no cache, refresh all prompt
    helper_refresh = RefreshIdxHelper_Mocked(RefreshIdxHelper_Mocked.POLICY_REFRESH_PREVIOUS_ALL)
# end

if use_cache_g and not use_refresh_g: # cache, but no refresh
    helper_refresh = RefreshIdxHelper_Mocked(RefreshIdxHelper_Mocked.POLICY_REFRESH_NONE)
# end

if use_cache_g and use_refresh_g:   # cache, and refresh
    with open(path_file_data_refresh, 'r') as file:
        data_refresh = json.load(file)
    # end
    helper_refresh = RefreshIdxHelper(data_refresh, type_hidden_str=type_hidden_refresh_str_g)
# end

'''some last variables'''
folder_report_g = 'ppl_result_abs_log_0309'
os.makedirs(folder_report_g, exist_ok=True)


'''start to run'''
# for budget_refresh_g in (1,2,4,8,16,32):
for budget_refresh_g in (32,):

    '''start to handle dataset based on hyper-parameter'''
    ds = ds_origin\
        .filter(lambda x: x["text"] is not None and len(x["text"].strip()) > 0)\
        .map(Tokenizer_wiki_simple(tokenizer, len_max_g), remove_columns=ds_origin.column_names)\
        .filter(lambda x: x["length"] >= len_max_g)\
        .sort("length")
    # end

    '''prepare dataloader'''
    loader = DataLoader(
        ds,
        batch_size=size_batch_g,
        shuffle=False,                 # keep sorted order
        collate_fn=Collater_wiki_simple(len_max_g, len_prompt_g, len_target_g, id_mask_g),
        drop_last=False
    )

    '''start the evaluation process'''
    for id_batch, batch in enumerate(tqdm(loader)):

        if id_batch < 90:
            continue

        result = run_model_with_budget(
            model,
            batch['ids_prompt_masked_full'].to(device_g),
            batch['ids_target_masked_full'].to(device_g),
            len_prompt_g,
            remasking='truth_top_k',
            steps=steps_g,
            gen_length=len_target_g,
            block_length=size_block_g,
            mask_id=id_mask_g,
            use_cache=use_cache_g,
            id_batch=id_batch,
            helper_refresh=helper_refresh,
            budget_refresh=budget_refresh_g
        )

        ppl, conf = calculate_ppl_and_conf(result[0], result[1])
        print(f'[{get_current_time_str()}] {ppl} | {conf}')
    # end for batch
# end for

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

  0%|          | 0/92 [00:00<?, ?it/s]

[90], [0]: tensor([124,  80,  10, 109,  70,  28, 121,  35,  19,  51,   2,  78,  67,  94,
         63, 114, 102, 125,  14, 118,  64,   0,  38, 127, 108,   9,  24, 110,
         33,  26,  60,  53])
[90], [1]: tensor([ 77, 154,  37,  43,  85,  68, 159, 132,  56,  44,  42,  50, 158, 110,
         13,  75, 140,  33,   6, 188, 168, 180,  79, 153,  36,  17, 157,  29,
        182,  48, 129, 165])
[90], [2]: tensor([152,  93,  78, 190, 201,  35,  41, 215, 232, 182,  67,  55, 165, 216,
        130, 244, 235, 185, 101, 208, 156, 103, 236,  50, 146, 179, 173,  89,
        211,  82, 149,  42])
[90], [3]: tensor([131, 150,  30, 234,  75, 232, 235, 238, 163, 151, 245,  24, 112, 239,
        241, 282, 114,  87,  14,  18,  20, 292,  36, 249, 217, 126, 186,  16,
         97, 258, 125,  41])


 99%|█████████▉| 91/92 [00:05<00:00, 15.75it/s]

[2026-03-10T12:47:30Z] 10.38774935646186 | 0.26641949541206933
[91], [0]: tensor([ 36,  25,  77,  37,  26,  96, 100,  53,  30,  19, 101,  46,  58,  99,
        111,  47,  85,  87,  38,  18,  91,  75, 119, 115, 107,  60,  16,  94,
        127,  27,  80,  24])
[91], [1]: tensor([148,  32, 100,  28,   4,  62, 131, 187,  76,   9,  71, 138, 140,  56,
         87, 114, 173, 179,  83, 151, 188, 159,  38,  47,  72, 153,  65, 160,
        128, 190, 133, 169])
[91], [2]: tensor([122, 207,  46, 160,  96, 164,  18,  63, 129, 184,  89, 216, 226, 235,
        143, 168, 246, 106,  83,  97, 121, 149, 177,   3, 230,   8, 183, 108,
        132, 255,  19, 233])
[91], [3]: tensor([198,  70,  31, 293, 202,  73, 290, 280, 289,  77,  43, 131, 220, 226,
        187,  16, 181, 112, 229, 122,  23,  21, 140,  76, 118, 150, 306, 264,
         10,  71, 179, 314])


100%|██████████| 92/92 [00:10<00:00,  8.41it/s]

[2026-03-10T12:47:35Z] 10.412957540507099 | 0.2764122139845908
